# B0 Validation (exploratory)
Full-volume evaluation of runs/full-b0-seed17/best.pt on the official
challenge validation folder. EXPLORATORY ONLY: no checkpoint selection,
threshold picking, or calibration on this data. Official validation
remains the internal 10% partition; note this folder's addition in the
data card.

In [ ]:
import sys, json, csv
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib import colormaps as mpl_cmaps

def find_repo_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "src" / "scanvidence").is_dir() or (p / "scanvidence").is_dir():
            return p
    return start.parent

project_root = find_repo_root()
for cand in (project_root / "src", project_root):
    sys.path.insert(0, str(cand))

CONFIG = {
    "checkpoint_path": "runs/full-b0-seed17/best.pt",
    "validation_data_root": r"C:\Users\NCC-HPC8\Desktop\BraTSGLI\data\ASNR-MICCAI-BraTS2023-GLI-Challenge-ValidationData\ASNR-MICCAI-BraTS2023-GLI-Challenge-ValidationData",
    "training_data_root": r"C:\Users\NCC-HPC8\Desktop\BraTSGLI\data\ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData\ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData",
    "widths": (16, 32, 64, 128), "num_classes": 4,
    "patch": 96, "stride": 48,
    "max_cases": 20,                 # None = every case with a mask
    "expected_params": 1599420,
    "use_official_validation_set": True,
    "seed": 17,
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("repo root:", project_root, "| device:", device)

In [ ]:
MOD_KEYS = ("seg", "t1n", "t1c", "t2w", "t2f")

def _modality_of(name):
    low = name.lower()
    for key in MOD_KEYS:
        for sep in ("-", "_"):
            for ext in (".nii.gz", ".nii"):
                if low.endswith(f"{sep}{key}{ext}"):
                    return key
    return None

def rglob_discover(root):
    groups = defaultdict(dict)
    for f in root.rglob("*"):
        if not f.is_file(): continue
        k = _modality_of(f.name)
        if not k: continue
        cid = f.name
        for k2 in MOD_KEYS:
            for sep in ("-", "_"):
                for ext in (".nii.gz", ".nii"):
                    suf = f"{sep}{k2}{ext}"
                    if cid.lower().endswith(suf): cid = cid[:-len(suf)]
        groups[cid][k] = f
    recs = []
    for cid, fm in sorted(groups.items()):
        recs.append({"patient_id": cid, **fm,
                     "seg_path": fm.get("seg"),
                     "available_sequences": [k for k in fm if k != "seg"]})
    return recs

def discover(root):
    root = Path(root)
    recs = rglob_discover(root)
    if not recs and (root / root.name).is_dir():      # duplicated-inner-folder layout
        recs = rglob_discover(root / root.name)
    if not recs:                                       # try repo dataset class last
        try:
            from scanvidence.data.datasets import BraTSDataset
            recs = BraTSDataset(str(root), track="GLI").discover()
        except Exception as e:
            print("BraTSDataset fallback failed:", e)
    needed = {"t1n", "t1c", "t2w", "t2f"}
    recs = [r for r in recs if needed <= set(r["available_sequences"])]
    return recs

root = CONFIG["validation_data_root"] if CONFIG["use_official_validation_set"] else CONFIG["training_data_root"]
val = discover(root)
with_gt = [r for r in val if r.get("seg_path") is not None]
print(f"cases with 4 modalities: {len(val)} | with ground truth: {len(with_gt)}")
if not with_gt:
    print("NO MASKS in this folder: metrics impossible here. Use it only to save predictions; run Dice on the internal val partition instead.")
cases = with_gt or val

In [ ]:
from scanvidence.models.backbone import SegResNetB0

model = SegResNetB0(in_channels=4, num_classes=CONFIG["num_classes"],
                    widths=CONFIG["widths"], dropout=0.0)
ckpt = torch.load(project_root / CONFIG["checkpoint_path"], map_location="cpu", weights_only=False)
state = ckpt.get("state_dict", ckpt.get("model", ckpt)) if isinstance(ckpt, dict) else ckpt
model.load_state_dict(state)
n_params = sum(p.numel() for p in model.parameters())
print(f"params {n_params} (expected {CONFIG['expected_params']}) | epoch {ckpt.get('epoch', '?')} | best_val_dice {ckpt.get('best_val_dice', '?')}")
assert n_params == CONFIG["expected_params"], "architecture/checkpoint mismatch — stop"
model.eval().to(device)

In [ ]:
def normalize_modality(v):                      # MUST match training: z-score on nonzero
    v = np.asarray(v, dtype=np.float32)
    m = v > 0
    out = np.zeros_like(v)
    if m.sum(): out[m] = (v[m] - v[m].mean()) / (v[m].std() + 1e-8)
    return out

def mod_path(rec, key):
    for cand in (rec.get(key), (rec.get("paths") or {}).get(key)):
        if cand: return Path(cand)
    raise KeyError(f"{key} missing for {rec['patient_id']}")

def load_case(rec):
    vol = np.stack([normalize_modality(nib.load(str(mod_path(rec, k))).get_fdata())
                    for k in ("t1n", "t1c", "t2w", "t2f")]).astype(np.float32)
    seg = nib.load(str(rec["seg_path"])).get_fdata().astype(np.int16) if rec.get("seg_path") else None
    if seg is not None:
        vals = set(np.unique(seg).tolist())
        assert vals <= {0, 1, 2, 3}, f"label drift {vals} — STOP"
    return vol, seg

def sliding_window_logits(model, vol, patch, stride, device):
    orig = vol.shape[1:]
    pad = tuple((patch - s % patch) % patch for s in orig)
    if any(pad):
        vol = np.pad(vol, ((0, 0), (0, pad[0]), (0, pad[1]), (0, pad[2])))
    _, D, H, W = vol.shape
    def starts(n):
        s = list(range(0, n - patch + 1, stride))
        if s[-1] != n - patch: s.append(n - patch)
        return s
    acc = np.zeros((4, D, H, W), np.float64); cnt = np.zeros((D, H, W), np.float64)
    with torch.no_grad():
        for i in starts(D):
            for j in starts(H):
                for k in starts(W):
                    x = torch.from_numpy(vol[:, i:i+patch, j:j+patch, k:k+patch])[None].to(device)
                    out = model(x)
                    if isinstance(out, (tuple, list)): out = out[0]
                    acc[:, i:i+patch, j:j+patch, k:k+patch] += out[0].cpu().numpy()
                    cnt[i:i+patch, j:j+patch, k:k+patch] += 1
    return (acc / cnt[None])[(slice(4),) + tuple(slice(0, o) for o in orig)]

In [ ]:
def region_masks(L):
    return {"ET": L == 3, "TC": (L == 1) | (L == 3), "WT": (L >= 1) & (L <= 3)}

def dice_empty_rule(a, b):
    a, b = a.astype(bool), b.astype(bool)
    if not a.any() and not b.any(): return 1.0
    if not a.any() or not b.any(): return 0.0
    return 2.0 * (a & b).sum() / (a.sum() + b.sum())

In [ ]:
rec = cases[0]
vol, seg = load_case(rec)
pred = sliding_window_logits(model, vol, CONFIG["patch"], CONFIG["stride"], device).argmax(0).astype(np.int16)
print(f"{rec['patient_id']}: predicted labels {np.unique(pred)}")

if seg is not None:
    gt_r, pd_r = region_masks(seg), region_masks(pred)
    for r in ("ET", "TC", "WT"):
        print(f"  {r} Dice {dice_empty_rule(pd_r[r], gt_r[r]):.4f}  (constant predictor: {1.0 if gt_r[r].sum()==0 else 0.0:.1f})")

    sums = seg.sum(axis=(1, 2)); si = int(np.argmax(sums)) if sums.max() > 0 else seg.shape[0] // 2
    cmap = mpl_cmaps["tab10"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(vol[3, si], cmap="gray"); axes[0].set_title("FLAIR"); axes[0].axis("off")
    axes[1].imshow(seg[si], cmap=cmap, vmin=0, vmax=3); axes[1].set_title("GT"); axes[1].axis("off")
    axes[2].imshow(pred[si], cmap=cmap, vmin=0, vmax=3); axes[2].set_title("B0 pred"); axes[2].axis("off")
    out_dir = project_root / "notebooks"; out_dir.mkdir(exist_ok=True)
    plt.tight_layout(); plt.savefig(out_dir / "b0_validation_slice.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
import csv, json
import nibabel as nib

out_dir = project_root / "notebooks"
out_dir.mkdir(exist_ok=True)

if not rows:
    print("="*60)
    print("NO GROUND TRUTH MASKS FOUND in the challenge validation folder.")
    print("This is expected: BraTS hides validation masks locally.")
    print("Dice/HD95 cannot be computed here. Saving raw predictions only.")
    print("="*60)
    
    pred_dir = out_dir / "predictions_challenge_val"
    pred_dir.mkdir(exist_ok=True)
    
    for rec in cases[:CONFIG["max_cases"] or len(cases)]:
        vol, _ = load_case(rec)
        logits = sliding_window_logits(model, vol, CONFIG["patch"], CONFIG["stride"], device)
        pred = logits.argmax(0).astype(np.int16)
        
        # Save as NIfTI using the first modality's affine matrix
        ref_img = nib.load(str(mod_path(rec, "t1n")))
        pred_img = nib.Nifti1Image(pred, ref_img.affine)
        nib.save(pred_img, pred_dir / f"{rec['patient_id']}_pred.nii.gz")
        
    print(f"Saved {len(cases[:CONFIG['max_cases'] or len(cases)])} prediction masks to {pred_dir}")
    print("Run the NEXT cell to evaluate metrics on your internal training split.")
else:
    with open(out_dir / "per_case.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys())
        w.writeheader()
        w.writerows(rows)
    
    summary = {"model": "SegResNetB0", "checkpoint": CONFIG["checkpoint_path"], "n_cases": len(rows),
               "device": str(device), "seed": CONFIG.get("seed", 17)}
    for r in ("ET", "TC", "WT"):
        vals = [x[f"dice_{r}"] for x in rows]
        const = [x[f"const_{r}"] for x in rows]
        summary[r] = {"mean": round(float(np.mean(vals)), 4), "std": round(float(np.std(vals)), 4),
                      "min": round(float(np.min(vals)), 4), "max": round(float(np.max(vals)), 4),
                      "constant_mean": round(float(np.mean(const)), 4)}
    (out_dir / "b0_validation_summary.json").write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))

In [ ]:
# Evaluate on Internal Training Data (Protocol-Correct Validation)
print("Switching to internal training data for metric evaluation...")
train_root = Path(CONFIG["training_data_root"])
train_cases = discover(train_root)
train_with_gt = [r for r in train_cases if r.get("seg_path") is not None]
print(f"Found {len(train_with_gt)} training cases with masks.")

# Take a sample of 20 cases for quick evaluation 
# (For the final Gate 1 report, use your exact frozen 10% val split IDs)
internal_val = train_with_gt[:20] 

internal_rows = []
for rec in internal_val:
    vol, seg = load_case(rec)
    pred = sliding_window_logits(model, vol, CONFIG["patch"], CONFIG["stride"], device).argmax(0).astype(np.int16)
    gt_r, pd_r = region_masks(seg), region_masks(pred)
    row = {"case": rec["patient_id"]}
    for r in ("ET", "TC", "WT"):
        row[f"dice_{r}"] = round(dice_empty_rule(pd_r[r], gt_r[r]), 4)
        row[f"const_{r}"] = 1.0 if gt_r[r].sum() == 0 else 0.0
    row["acc"] = round(float((pred == seg).mean()), 4)
    internal_rows.append(row)
    print(f"{row['case']}: ET {row['dice_ET']:.3f} TC {row['dice_TC']:.3f} WT {row['dice_WT']:.3f}")

# Summary
import numpy as np
summary = {"model": "SegResNetB0", "checkpoint": CONFIG["checkpoint_path"], "n_cases": len(internal_rows), "device": str(device)}
for r in ("ET", "TC", "WT"):
    vals = [x[f"dice_{r}"] for x in internal_rows]
    const = [x[f"const_{r}"] for x in internal_rows]
    summary[r] = {"mean": round(float(np.mean(vals)), 4), "std": round(float(np.std(vals)), 4),
                  "min": round(float(np.min(vals)), 4), "max": round(float(np.max(vals)), 4),
                  "constant_mean": round(float(np.mean(const)), 4)}

out_dir = project_root / "notebooks"
(out_dir / "b0_internal_val_summary.json").write_text(json.dumps(summary, indent=2))
print("\nINTERNAL VALIDATION SUMMARY:")
print(json.dumps(summary, indent=2))